In [1]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd

# Các thư viện tối ưu và thuật toán Machine Learning
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# 4 Mô hình theo yêu cầu của bạn
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Tắt cảnh báo phiền phức
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 20  # Tăng lên 50 nếu bạn muốn thuật toán tìm kiếm kỹ hơn
ARTIFACTS_DIR = "artifacts/tuning"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

c:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def find_file_automatically(filename):
    """Hàm tự động quét tìm file trong thư mục hiện tại và các thư mục con/cha"""
    if os.path.exists(filename):
        return filename
    
    parent_path = os.path.join("..", filename)
    if os.path.exists(parent_path):
        return parent_path
        
    for root, dirs, files in os.walk("."):
        if filename in files:
            return os.path.join(root, filename)
            
    for root, dirs, files in os.walk(".."):
        if filename in files:
            return os.path.join(root, filename)
            
    return None

# Tự động tìm file
target_filename = "cleaned_train.csv"
file_path = find_file_automatically(target_filename)

if file_path is None:
    print(f"❌ KHÔNG TÌM THẤY FILE '{target_filename}'!")
    print(f"Thư mục Python đang đứng là: {os.getcwd()}")
    raise FileNotFoundError(f"Vui lòng đặt file {target_filename} vào thư mục dự án.")
else:
    print(f"✅ Đã tìm thấy file tại đường dẫn: {file_path}")
    df = pd.read_csv(file_path)

    # Tách đặc trưng X và nhãn target y
    TARGET_COL = "Price"
    if TARGET_COL not in df.columns:
        raise KeyError(f"Không tìm thấy cột mục tiêu '{TARGET_COL}' trong file dữ liệu.")

    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL]

    print(f"-> Kích thước tập dữ liệu: {X.shape[0]} mẫu, {X.shape[1]} đặc trưng.")

✅ Đã tìm thấy file tại đường dẫn: ..\data\processed\cleaned_train.csv
-> Kích thước tập dữ liệu: 30995 mẫu, 59 đặc trưng.


In [3]:
def evaluate_model_cv_rmse(model_class, params, X, y):
    """
    Hàm chia dữ liệu thành 5-Fold, huấn luyện và trả về 
    điểm RMSE trung bình trên tập Validation.
    """
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rmse_list = []
    
    X_arr = X.values if isinstance(X, pd.DataFrame) else X
    y_arr = y.values if isinstance(y, pd.Series) else y
    
    for train_idx, val_idx in kf.split(X_arr):
        X_train, X_val = X_arr[train_idx], X_arr[val_idx]
        y_train, y_val = y_arr[train_idx], y_arr[val_idx]
        
        model = model_class(**params, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        
        preds = model.predict(X_val)
        
        # Tính RMSE = Căn bậc hai của MSE
        mse = mean_squared_error(y_val, preds)
        rmse = np.sqrt(mse)
        rmse_list.append(rmse)
        
    return np.mean(rmse_list)

In [4]:
def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200, step=50),
        "max_depth": trial.suggest_int("max_depth", 10, 25, step=5),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 4),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),
        "n_jobs": -1
    }
    # Gọi hàm đánh giá dựa trên RMSE
    return evaluate_model_cv_rmse(RandomForestRegressor, params, X, y)

print("🚀 Đang Tuning Random Forest dựa trên chỉ số RMSE...")
rf_study = optuna.create_study(direction="minimize")
rf_study.optimize(rf_objective, n_trials=N_TRIALS)
print(f"🏆 Random Forest - Best RMSE: {rf_study.best_value:,.2f}")

🚀 Đang Tuning Random Forest dựa trên chỉ số RMSE...
🏆 Random Forest - Best RMSE: 0.65


In [7]:
def gb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0)
    }
    return evaluate_model_cv_rmse(GradientBoostingRegressor, params, X, y)

print("🚀 Đang Tuning Gradient Boosting dựa trên chỉ số RMSE...")
gb_study = optuna.create_study(direction="minimize")
gb_study.optimize(gb_objective, n_trials=N_TRIALS)
print(f"🏆 Gradient Boosting - Best RMSE: {gb_study.best_value:,.2f}")

🚀 Đang Tuning Gradient Boosting dựa trên chỉ số RMSE...
🏆 Gradient Boosting - Best RMSE: 0.66


In [8]:
def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_jobs": -1,
        "verbose": -1
    }
    return evaluate_model_cv_rmse(LGBMRegressor, params, X, y)

print("🚀 Đang Tuning LightGBM dựa trên chỉ số RMSE...")
lgbm_study = optuna.create_study(direction="minimize")
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS)
print(f"🏆 LightGBM - Best RMSE: {lgbm_study.best_value:,.2f}")

🚀 Đang Tuning LightGBM dựa trên chỉ số RMSE...
🏆 LightGBM - Best RMSE: 0.64


In [10]:
def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
        "n_jobs": -1
    }
    return evaluate_model_cv_rmse(XGBRegressor, params, X, y)

print("🚀 Đang Tuning XGBoost dựa trên chỉ số RMSE...")
xgb_study = optuna.create_study(direction="minimize")
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS)
print(f"🏆 XGBoost - Best RMSE: {xgb_study.best_value:,.2f}")

🚀 Đang Tuning XGBoost dựa trên chỉ số RMSE...
🏆 XGBoost - Best RMSE: 0.65


In [11]:
print("=" * 70)
print("BẢNG XẾP HẠNG KẾT QUẢ TUNING (THEO RMSE THẤP NHẤT)")
print("=" * 70)

results = [
    {"Model": "Random Forest", "Best RMSE": rf_study.best_value, "Params": rf_study.best_params},
    {"Model": "Gradient Boosting", "Best RMSE": gb_study.best_value, "Params": gb_study.best_params},
    {"Model": "XGBoost", "Best RMSE": xgb_study.best_value, "Params": xgb_study.best_params},
    {"Model": "LightGBM", "Best RMSE": lgbm_study.best_value, "Params": lgbm_study.best_params}
]

# Lưu kết quả vào Dataframe và sắp xếp tăng dần theo điểm số RMSE
summary_df = pd.DataFrame(results).sort_values(by="Best RMSE").reset_index(drop=True)
summary_df.index += 1

for idx, row in summary_df.iterrows():
    print(f"#{idx}  {row['Model']:20s}  👉  Best RMSE = {row['Best RMSE']:>12,.2f}")

# Xuất bộ tham số tốt nhất tối ưu theo RMSE ra file JSON
for item in results:
    model_name_slug = item["Model"].lower().replace(" ", "_")
    output_path = os.path.join(ARTIFACTS_DIR, f"best_params_{model_name_slug}.json")
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(item["Params"], f, indent=4, ensure_ascii=False)

print(f"\n💾 Lưu tham số tốt nhất tối ưu theo RMSE tại: '{ARTIFACTS_DIR}/'")
print(f"🏆 Kết luận mô hình tối ưu nhất: {summary_df.iloc[0]['Model']}")

BẢNG XẾP HẠNG KẾT QUẢ TUNING (THEO RMSE THẤP NHẤT)
#1  LightGBM              👉  Best RMSE =         0.64
#2  XGBoost               👉  Best RMSE =         0.65
#3  Random Forest         👉  Best RMSE =         0.65
#4  Gradient Boosting     👉  Best RMSE =         0.66

💾 Lưu tham số tốt nhất tối ưu theo RMSE tại: 'artifacts/tuning/'
🏆 Kết luận mô hình tối ưu nhất: LightGBM


In [12]:
import joblib

# 1. Định nghĩa và tạo thư mục lưu trữ mới theo yêu cầu của bạn
TUNED_MODELS_DIR = "artifacts/tuned_models"
os.makedirs(TUNED_MODELS_DIR, exist_ok=True)

print("=" * 70)
print(f"🔄 ĐANG TIẾN HÀNH RETRAIN VÀ LƯU MÔ HÌNH VÀO '{TUNED_MODELS_DIR}/'")
print("=" * 70)

# Cấu hình danh sách ánh xạ các đối tượng class mô hình và kết quả nghiên cứu tương ứng
models_to_save = [
    {"name": "Random_Forest", "class": RandomForestRegressor, "study": rf_study},
    {"name": "Gradient_Boosting", "class": GradientBoostingRegressor, "study": gb_study},
    {"name": "XGBoost", "class": XGBRegressor, "study": xgb_study},
    {"name": "LightGBM", "class": LGBMRegressor, "study": lgbm_study}
]

# Chuyển đổi dữ liệu sang dạng numpy để đảm bảo không bị lỗi tương thích cấu trúc định dạng nền tảng
X_final = X.values if isinstance(X, pd.DataFrame) else X
y_final = y.values if isinstance(y, pd.Series) else y

for item in models_to_save:
    model_name = item["name"]
    model_class = item["class"]
    best_params = item["study"].best_params # Lấy bộ tham số tối ưu nhất từ Optuna
    
    print(f"⏳ Đang huấn luyện lại {model_name} với bộ tham số tốt nhất...")
    
    # Khởi tạo mô hình dựa trên best_params tìm được
    if model_name in ["Random_Forest", "XGBoost", "LightGBM"]:
        # Các mô hình hỗ trợ đa nhân song song n_jobs=-1 để chạy cho nhanh
        final_model = model_class(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        final_model = model_class(**best_params, random_state=RANDOM_STATE)
        
    # Huấn luyện trên toàn bộ tập dữ liệu sạch ban đầu
    final_model.fit(X_final, y_final)
    
    # Định nghĩa đường dẫn lưu file (.pkl)
    model_file_path = os.path.join(TUNED_MODELS_DIR, f"tuned_{model_name.lower()}.pkl")
    
    # Thực hiện lưu mô hình đóng gói vật lý xuống đĩa cứng
    joblib.dump(final_model, model_file_path)
    print(f"✅ Đã lưu thành công file mô hình: {model_file_path}")

print("\n🎉 HOÀN THÀNH: Tất cả 4 mô hình đã được tuned và đóng gói gọn gàng sẵn sàng sử dụng!")

🔄 ĐANG TIẾN HÀNH RETRAIN VÀ LƯU MÔ HÌNH VÀO 'artifacts/tuned_models/'
⏳ Đang huấn luyện lại Random_Forest với bộ tham số tốt nhất...
✅ Đã lưu thành công file mô hình: artifacts/tuned_models\tuned_random_forest.pkl
⏳ Đang huấn luyện lại Gradient_Boosting với bộ tham số tốt nhất...
✅ Đã lưu thành công file mô hình: artifacts/tuned_models\tuned_gradient_boosting.pkl
⏳ Đang huấn luyện lại XGBoost với bộ tham số tốt nhất...
✅ Đã lưu thành công file mô hình: artifacts/tuned_models\tuned_xgboost.pkl
⏳ Đang huấn luyện lại LightGBM với bộ tham số tốt nhất...
✅ Đã lưu thành công file mô hình: artifacts/tuned_models\tuned_lightgbm.pkl

🎉 HOÀN THÀNH: Tất cả 4 mô hình đã được tuned và đóng gói gọn gàng sẵn sàng sử dụng!
